In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=False)
required_keys = ("GROQ_API_KEY", "TAVILY_API_KEY")
missing_keys = [key for key in required_keys if not os.environ.get(key)]

if missing_keys:
    raise EnvironmentError(f"Missing API keys: {', '.join(missing_keys)}")

from pathlib import Path

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.environ["GROQ_API_KEY"],
)
web_search = TavilySearch(max_results=5)

print("LLM and web search tool are ready.")

LLM and web search tool are ready.


In [2]:
from langchain_docling import DoclingLoader

FILE_PATH = Path("attention_is_all_you_need.pdf")

documents = DoclingLoader(file_path=str(FILE_PATH)).load()

print(f"Loaded {len(documents)} document sections from {FILE_PATH.name}.")

c:\Users\anvek\OneDrive\Documents\Desktop\SCFP125011-3rd-Sem\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 2680.65it/s]
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (963 > 512). Running this sequence through the model will re

Loaded 67 document sections from attention_is_all_you_need.pdf.


In [3]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(documents)
if not chunks:
    raise ValueError("The document did not produce any searchable text chunks.")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = Chroma(
    collection_name="document_knowledge_base",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)
vectorstore.reset_collection()

print(f"Split the document into {len(chunks)} chunks. Vector store is ready.")

c:\Users\anvek\OneDrive\Documents\Desktop\SCFP125011-3rd-Sem\myenv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\anvek\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\anvek\OneDrive\Documents\Desktop\SCFP125011-3rd-Sem\my

Split the document into 75 chunks. Vector store is ready.


In [4]:
# Chroma accepts only scalar metadata values. Docling stores its detailed
# provenance under dl_meta, so extract source/page fields and omit nested data.
indexable_chunks = []
for chunk in chunks:
    raw_metadata = chunk.metadata
    metadata = {
        key: value
        for key, value in raw_metadata.items()
        if isinstance(value, (str, int, float, bool))
    }

    docling_metadata = raw_metadata.get("dl_meta")
    if isinstance(docling_metadata, dict):
        origin = docling_metadata.get("origin")
        if isinstance(origin, dict):
            source = origin.get("filename")
            if isinstance(source, str):
                metadata["source"] = source

        page_numbers = {
            provenance.get("page_no")
            for item in docling_metadata.get("doc_items", [])
            if isinstance(item, dict)
            for provenance in item.get("prov", [])
            if isinstance(provenance, dict)
            and isinstance(provenance.get("page_no"), int)
        }
        if page_numbers:
            metadata["pages"] = ", ".join(
                str(page) for page in sorted(page_numbers)
            )

    indexable_chunks.append(
        chunk.model_copy(update={"metadata": metadata})
    )

vectorstore.add_documents(indexable_chunks)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"Indexed {len(indexable_chunks)} chunks in the local Chroma knowledge base.")

Indexed 75 chunks in the local Chroma knowledge base.


In [5]:
@tool
def search_knowledge_base(query: str) -> str:
    """Search the uploaded document knowledge base for relevant passages.

    Use this for questions about facts, explanations, or details in the user's
    uploaded documents. The query should describe the information to find.
    """
    matches = retriever.invoke(query)
    if not matches:
        return "No relevant passages were found in the document knowledge base."

    passages = []
    for number, doc in enumerate(matches, start=1):
        source = doc.metadata.get("source") or doc.metadata.get("file_path") or FILE_PATH.name
        page = doc.metadata.get("page")
        if page is None:
            page = doc.metadata.get("pages")
        page_label = f", page {page}" if page is not None else ""
        passages.append(
            f"Passage {number} (source: {source}{page_label}):\n{doc.page_content}"
        )
    return "\n\n".join(passages)

print("Tool name:", search_knowledge_base.name)

Tool name: search_knowledge_base


In [6]:
print(search_knowledge_base.invoke("What is self-attention?"))

Passage 1 (source: attention_is_all_you_need.pdf, page 2):
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations [4, 27, 28, 22].

Passage 2 (source: attention_is_all_you_need.pdf, page 7):
As side benefit, self-attention could yield more interpretable models. We inspect attention distributions from our models and present and discuss examples in the appendix. Not only do individual attention heads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntactic and semantic structure of the sentences.

Passage 3 (source: attention_is_all_you_need.pdf, page 3):
3.2 Attention
An attention function can be described as mapping a query and 

In [7]:
agent = create_agent(
    model=llm,
    tools=[web_search, search_knowledge_base],
    system_prompt=(
        "You are a helpful research assistant. For questions about the user's uploaded "
        "documents, search the document knowledge base and ground your answer in the "
        "retrieved passages. For recent or general external information, use web search. "
        "Use both sources when appropriate. Be clear about which source supports your "
        "answer, cite document source/page metadata when available, and do not invent facts."
    ),
)

print("Agent is ready.")

Agent is ready.


In [8]:
def ask(question):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})

    # Show which tools the agent decided to call
    tools_used = [
        call["name"]
        for message in result["messages"]
        for call in getattr(message, "tool_calls", [])
    ]
    print("Question:", question)
    print("Tools used:", tools_used if tools_used else "none")
    print("\nAnswer:\n", result["messages"][-1].content)

ask("What is multi-head attention, and why did the authors use it instead of a single attention function?")

Question: What is multi-head attention, and why did the authors use it instead of a single attention function?
Tools used: ['search_knowledge_base']

Answer:
 **Multi‑head attention** is a variant of the scaled‑dot‑product attention mechanism in which several “heads” operate in parallel.  
For each head \(i\) the input vectors are linearly projected into query, key and value spaces using separate weight matrices
\(W_Q^{(i)}\), \(W_K^{(i)}\) and \(W_V^{(i)}\).  
Each head then performs the standard scaled‑dot‑product attention:

\[
\text{Attention}(Q_i,K_i,V_i)=\text{softmax}\!\left(\frac{Q_iK_i^{\top}}{\sqrt{d_k}}\right)V_i .
\]

The outputs of all heads are concatenated and finally projected back to the model dimension with a matrix \(W_O\).  
This is described in the paper (page 5, section 3.2.2) where the authors state that they use \(h=8\) heads, each with
\(d_k=d_v=d_{\text{model}}/h=64\); the reduced dimensionality per head keeps the total computational cost comparable to a singl

In [9]:
ask("What are the latest news about large language models?")

Question: What are the latest news about large language models?
Tools used: ['tavily_search']

Answer:
 **Recent headlines on large‑language models (LLMs) – November 2026**

| Date | Source | Key story | Take‑away |
|------|--------|-----------|-----------|
| 12 Nov 2026 | *Scienmag* (via Discover AI) | Bournemouth University releases the **PT‑LLM‑8** psychometric scale that measures users’ perceived trust in LLMs (truthfulness, safety, privacy, etc.). | A new, validated tool for researchers and designers to quantify how trustworthy people think AI systems are. |
| 13 Nov 2026 | *Storyboard18* | ByteDance’s chief Zhang Yiming announces plans for a **Meta‑Muse‑style AI assistant** and a $70 B AI investment, stressing the need for larger training datasets and “long‑term R&D” over model distillation. | ByteDance is positioning itself to catch up with global LLM leaders, but will focus on data scale rather than shortcut techniques. |
| 12 Nov 2026 | *TechXplore* (Physical Review Applied) |

In [10]:
ask("What is 15 multiplied by 12?")

Question: What is 15 multiplied by 12?
Tools used: none

Answer:
 15 multiplied by 12 equals **180**.
